# 03 · Transformer (DeiT-Small)

Vision Transformer pretrained on ImageNet, fully fine-tuned. DeiT-Small (22M params) is close in size to ResNet-50, so CNN vs Transformer is a fair comparison.

Rules: same settings for every `VARIANT`; choose on **val**; evaluate **test** once.
Results go to `results/transformer/<variant>__<model>__s<seed>.json` and `_preds.csv`.

In [ ]:
VARIANT = "original"     # original, clahe, histeq, gsam, gsam_crop, gsam_light, gsam_light_crop, sam, sam_crop
MODEL_NAME = "deit_small_patch16_224.fb_in1k"   # timm name; e.g. deit_small_patch16_224.fb_in1k, vit_base_patch16_224
EPOCHS = 15
LR = 5e-5
BATCH_SIZE = 64
NUM_WORKERS = 2
SEED = 0

In [ ]:
from pathlib import Path

# ---- paths (preset) -------------------------------------------------------
# project root = the folder that contains pipeline_preprocess/ (works from any sub-folder)
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_preprocess").is_dir())
DATA_ROOT = PROJECT_ROOT / "data" / "preprocessing_data"      # original_data/, clahe_data/, gsam_data/, ...
RESULTS_DIR = PROJECT_ROOT / "results" / "transformer"

# Colab + Google Drive: uncomment and point to the shared folder
# PROJECT_ROOT = Path("/content/drive/MyDrive/dip-project")
# DATA_ROOT = PROJECT_ROOT / "data" / "preprocessing_data"
# RESULTS_DIR = PROJECT_ROOT / "results" / "transformer"

data_dir = DATA_ROOT / f"{VARIANT}_data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
assert data_dir.is_dir(), f"{data_dir} not found. Available: {sorted(p.name for p in DATA_ROOT.glob('*_data'))}"
print("data   :", data_dir, sorted(p.name for p in data_dir.iterdir()))
print("results:", RESULTS_DIR)

In [ ]:
import numpy as np, torch, torch.nn as nn, timm, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms as T
from sklearn.metrics import f1_score

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet (pretrained weights)
FILL = (124, 116, 104)                                     # same grey as the segmentation background

# augmentation: TRAIN only, geometric only (colour jitter would blur the CLAHE / hist-eq comparison)
train_tf = T.Compose([T.RandomRotation(30, fill=FILL), T.RandomResizedCrop(224, scale=(0.6, 1.0)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = datasets.ImageFolder(data_dir / "train", train_tf)
val_ds = datasets.ImageFolder(data_dir / "val", eval_tf)
test_ds = datasets.ImageFolder(data_dir / "test", eval_tf)
CLASSES = train_ds.classes
test_paths = [Path(s).relative_to(data_dir).as_posix() for s, _ in test_ds.samples]
dl = lambda ds, shuffle: DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS)
train_dl, val_dl, test_dl = dl(train_ds, True), dl(val_ds, False), dl(test_ds, False)
print(device, {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)})

## Model

In [ ]:
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(device)
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params")

## Train: keep the best epoch by val macro-F1

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_dl))
crit = nn.CrossEntropyLoss(label_smoothing=0.1)

@torch.no_grad()
def predict(loader):
    model.eval(); ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(device)).argmax(1).cpu()); ys.append(y)
    return torch.cat(ys).numpy(), torch.cat(ps).numpy()

history, val_f1 = [], -1
for ep in range(1, EPOCHS + 1):
    model.train(); losses = []
    for x, y in train_dl:
        loss = crit(model(x.to(device)), y.to(device))
        opt.zero_grad(); loss.backward(); opt.step(); sched.step(); losses.append(loss.item())
    f1 = f1_score(*predict(val_dl), average="macro")
    history.append((np.mean(losses), f1))
    print(f"ep {ep:2d}  loss {np.mean(losses):.3f}  val_f1 {f1:.3f}")
    if f1 > val_f1:
        val_f1, best_epoch = f1, ep
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot([h[0] for h in history]); ax[0].set_title("train loss")
ax[1].plot([h[1] for h in history]); ax[1].set_title("val macro-F1"); ax[1].axvline(best_epoch - 1, ls="--", c="grey")
plt.show()
CONFIG = {"epochs": EPOCHS, "best_epoch": best_epoch, "lr": LR, "batch_size": BATCH_SIZE, "val_f1": val_f1}

## Test (once, best-val checkpoint)

In [ ]:
model.load_state_dict(best_state)
y_true, y_pred = predict(test_dl)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
test_metrics = {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f1}
print(f"{VARIANT} / {MODEL_NAME}:", {k: round(v, 4) for k, v in test_metrics.items()})
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASSES, xticks_rotation=60,
                                        normalize="true", values_format=".2f", colorbar=False)
plt.title(f"{VARIANT} / {MODEL_NAME}"); plt.tight_layout(); plt.show()

In [ ]:
import json, pandas as pd
run = f"{VARIANT}__{MODEL_NAME}__s{SEED}"
pd.DataFrame({"path": test_paths, "y_true": [CLASSES[i] for i in y_true], "y_pred": [CLASSES[i] for i in y_pred]}
            ).to_csv(RESULTS_DIR / f"{run}_preds.csv", index=False)
json.dump({"variant": VARIANT, "model": MODEL_NAME, "seed": SEED, **test_metrics, **CONFIG},
          open(RESULTS_DIR / f"{run}.json", "w"), indent=2)
print("saved:", RESULTS_DIR / run)